# Practicum 2 — executive compensation

This notebook estimates the lower-truncated output distributions, solves the
incentive-compatibility equation, fits the optimal wage schedule, decomposes
moral-hazard costs, and evaluates the double-gamma counterfactual.

In [1]:
from pathlib import Path

import pandas as pd

from dse_practicum import P2FitConfig, fit_practicum2

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "practicum" / "practicum2" / "data").exists()
)

DATA = ROOT / "practicum" / "practicum2" / "data"
DATA

PosixPath('/Users/bc3139/repo/summer2026/DSE2026/practicum/practicum2/data')

## 1. Fast point estimates

`bootstrap_reps=0` returns fast analytic standard-error approximations.

In [2]:
quick_result = fit_practicum2(
    DATA / "simulated_moral_hazard_data.dta",
    config=P2FitConfig(n_starts=12, bootstrap_reps=0),
)
quick_result.summary()

,estimate
quantity,
psi,-1.441944
mu_w,-0.051451
sigma,0.713388
mu_s,-0.285481
gamma,0.000102
alpha,3.338030
beta,2.832603
se_mu_w,0.008849
se_sigma,0.007053


In [3]:
pd.Series(quick_result.diagnostics, name="value").to_frame()

,value
n_observations,8000
stage1_converged,True
stage2_converged,True
stage2_nfev,15
wage_rmse,1738.263063
bootstrap_successes,0
cf_delta_1_ratio,0.5
cf_delta_2_ratio,0.5
cf_delta_3_difference,0.0


The double-gamma invariants provide a strong internal check:

- counterfactual delta 1 / baseline delta 1 = 0.5;
- counterfactual delta 2 / baseline delta 2 = 0.5;
- counterfactual delta 3 − baseline delta 3 = 0.

In [4]:
{
    "delta_1_ratio": quick_result.diagnostics["cf_delta_1_ratio"],
    "delta_2_ratio": quick_result.diagnostics["cf_delta_2_ratio"],
    "delta_3_difference": quick_result.diagnostics[
        "cf_delta_3_difference"
    ],
}

{'delta_1_ratio': 0.5, 'delta_2_ratio': 0.5, 'delta_3_difference': 0.0}

## 2. Final inference with a two-stage bootstrap

Start with 25 repetitions to check runtime. Use 100–500 for the final
submission. Each resample re-estimates both stages.

In [5]:
# final_result = fit_practicum2(
#     DATA / "simulated_moral_hazard_data.dta",
#     config=P2FitConfig(
#         n_starts=12,
#         bootstrap_reps=200,
#         bootstrap_starts=2,
#     ),
# )
# final_result.summary()

Until the bootstrap cell is run, use the quick result.

In [6]:
result = quick_result

## 3. Create the submission file

In [7]:
submission = result.to_submission(
    DATA / "sample_submission.csv",
    ROOT / "practicum" / "practicum2_submission.csv",
)
submission

,ID,ESTIMATE
0,psi,-1.441944
1,mu_w,-0.051451
2,sigma,0.713388
3,mu_s,-0.285481
4,gamma,0.000102
5,alpha,3.338030
6,beta,2.832603
7,se_mu_w,0.008849
8,se_sigma,0.007053
9,se_mu_s,0.018820
